# Distributed PyTorch Training with Ray Train

This notebook compares ordinary single-process PyTorch training with synchronous data-parallel training using Ray Train's `TorchTrainer`. FashionMNIST and a small fully connected network keep the example approachable, while the benchmark highlights an important lesson: adding workers does not automatically reduce runtime. Dataset setup, CPU threading, global batch size, gradient synchronization, and Ray startup overhead all affect the result.

## Learning objectives

By the end of this notebook, you will be able to:

- Build and time a single-process PyTorch training loop.
- Use `prepare_model()` and `prepare_data_loader()` for distributed training.
- Explain how a `DistributedSampler` partitions data across workers.
- Distinguish per-worker batch size from global batch size.
- Diagnose why a small CPU workload may not speed up with multiple Ray workers.
- Design a fairer distributed-training benchmark.

## 1. Single-worker baseline

The baseline is intentionally limited to one PyTorch CPU thread. This makes the experiment a one-CPU versus four-CPU scale-out comparison. Without that limit, the nominally single-worker process can use many intra-operation threads and may already consume most of the machine.

In [ ]:
import os
import time
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

DATA_ROOT = Path("data").resolve()
NUM_EPOCHS = 3
BATCH_SIZE_PER_WORKER = 64
LEARNING_RATE = 0.01


def get_dataset(*, download: bool = False):
    return datasets.FashionMNIST(
        root=DATA_ROOT,
        train=True,
        transform=v2.Compose([
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
        ]),
        download=download,
    )


# Download once in the notebook process. Ray workers only read cached files.
_ = get_dataset(download=True)


class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        return self.linear_relu_stack(self.flatten(x))


def train_single_worker() -> float:
    dataset = get_dataset(download=False)
    dataloader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE_PER_WORKER,
        shuffle=True,
    )

    model = NeuralNetwork()

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE)

    for epoch in range(NUM_EPOCHS):
        for inputs, labels in dataloader:
            optimizer.zero_grad(set_to_none=True)
            predictions = model(inputs)
            loss = criterion(predictions, labels)
            loss.backward()
            optimizer.step()
        print(f"single epoch={epoch + 1}, final_batch_loss={loss.item():.4f}")

    return loss.item()


# PyTorch otherwise uses several CPU threads inside this one process.
torch.set_num_threads(1)
print(f"Single-worker PyTorch threads: {torch.get_num_threads()}")

start = time.perf_counter()
single_final_loss = train_single_worker()
single_worker_seconds = time.perf_counter() - start

print(f"Single-worker training time: {single_worker_seconds:.2f} seconds")

### What the baseline measures

FashionMNIST contains 60,000 training samples. With a batch size of 64, the single worker performs approximately $\lceil 60{,}000 / 64 \rceil = 938$ optimizer steps per epoch. The timer starts after the dataset has been downloaded, so it measures dataset loading from local storage and training but excludes the one-time network download.

The printed loss is the final batch's loss, not the average epoch loss. It is sufficient for demonstrating the training loop, but production experiments should aggregate loss across the epoch and evaluate on a validation set.

## 2. Distributed multi-worker training

Ray starts four training processes on the local machine. `ray.train.torch.prepare_model()` wraps the network with PyTorch `DistributedDataParallel` (DDP), while `prepare_data_loader()` installs a `DistributedSampler` so each worker receives a different quarter of the dataset.

Every worker still holds a complete model replica. After each backward pass, DDP performs a collective operation to average gradients across all replicas before their optimizers step. This synchronization keeps the models identical, but it adds communication and waiting time.

In [ ]:
import ray
import ray.train
import ray.train.torch
from ray.train import ScalingConfig
from ray.train.torch import TorchTrainer

def train_func_distributed():
    context = ray.train.get_context()
    world_size = context.get_world_size()
    rank = context.get_world_rank()

    # Each Ray worker reserves one CPU, so prevent hidden thread oversubscription.
    torch.set_num_threads(1)

    # The driver downloaded FashionMNIST before workers were started.
    dataset = get_dataset(download=False)
    dataloader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE_PER_WORKER,
        shuffle=True,
    )
    dataloader = ray.train.torch.prepare_data_loader(dataloader)

    model = NeuralNetwork()
    model = ray.train.torch.prepare_model(model)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE)

    if rank == 0:
        print(
            f"world_size={world_size}, "
            f"local_batch_size={BATCH_SIZE_PER_WORKER}, "
            f"global_batch_size={BATCH_SIZE_PER_WORKER * world_size}, "
            f"torch_threads_per_worker={torch.get_num_threads()}, "
            f"OMP_NUM_THREADS={os.environ.get('OMP_NUM_THREADS')}"
        )

    for epoch in range(NUM_EPOCHS):
        if world_size > 1:
            dataloader.sampler.set_epoch(epoch)

        for inputs, labels in dataloader:
            optimizer.zero_grad(set_to_none=True)
            predictions = model(inputs)
            loss = criterion(predictions, labels)
            loss.backward()
            optimizer.step()

        metrics = {
            "epoch": epoch + 1,
            "final_batch_loss": loss.item(),
            "world_size": world_size,
            "global_batch_size": BATCH_SIZE_PER_WORKER * world_size,
        }
        ray.train.report(metrics)
        if rank == 0:
            print(
                f"distributed epoch={epoch + 1}, "
                f"final_batch_loss={loss.item():.4f}"
            )


# Start one four-CPU Ray runtime for this benchmark.
ray.shutdown()
ray.init(num_cpus=4, include_dashboard=False, logging_level="ERROR")

trainer = TorchTrainer(
    train_loop_per_worker=train_func_distributed,
    scaling_config=ScalingConfig(
        num_workers=4,
        use_gpu=False,
        resources_per_worker={"CPU": 1},
    ),
)

# This end-to-end timer includes worker and process-group startup.
start = time.perf_counter()
result = trainer.fit()
distributed_seconds = time.perf_counter() - start

print(f"Distributed training time: {distributed_seconds:.2f} seconds")
ray.shutdown()


In [ ]:
speedup = single_worker_seconds / distributed_seconds

print(f"Single worker:       {single_worker_seconds:.2f} seconds")
print(f"Four Ray workers:    {distributed_seconds:.2f} seconds")
print(f"Observed speedup:     {speedup:.2f}x")

if speedup <= 1:
    print("Ray was not faster for this small CPU-bound workload.")

## 3. Interpreting the comparison

With four workers, each `DistributedSampler` shard contains approximately 15,000 samples. At 64 samples per worker, each process performs approximately $\lceil 15{,}000 / 64 \rceil = 235$ steps per epoch. The effective global batch is $64 \times 4 = 256$, and DDP synchronizes gradients after each of those 235 steps.

This is a throughput comparison, not an optimization-equivalent comparison. The single worker performs about 938 optimizer updates per epoch with a global batch of 64, whereas the distributed run performs about 235 updates with a global batch of 256. Keeping a global batch of 64 would require 16 samples per worker and about 938 synchronization points per epoch, which is likely slower for this small model.

### Why four workers may take the same time—or longer

1. **The model is small.** The network has roughly 670,000 parameters, so each CPU forward and backward pass is inexpensive. DDP must nevertheless synchronize about 2.7 MB of float32 gradients after every step. Communication and barrier waiting can therefore rival the useful computation.
2. **Ray has fixed startup costs.** `trainer.fit()` creates worker processes, establishes the distributed process group, serializes the training function, and later tears the workers down. Three short epochs may not run long enough to amortize those costs.
3. **A single PyTorch process can already be multithreaded.** On the machine used to investigate this notebook, PyTorch defaulted to 14 intra-operation threads. The revised baseline explicitly uses one thread so it does not silently receive more CPU resources than the four one-CPU Ray workers.
4. **Input setup can dominate.** The original function used `download=True` inside every worker, and the logs showed four concurrent 26.4 MB downloads lasting several minutes. This notebook now downloads once on the driver and uses the cached files in workers.
5. **Stale Ray runtimes can interfere.** The original logs reported multiple active Ray instances. If that occurs again, stop them from a terminal with `ray stop --force`, restart the notebook kernel, and rerun from the top.

### Designing a fair benchmark

There are two valid but different comparisons:

- **Resource-scaling comparison:** use one CPU thread and batch size 64 for the baseline, then four one-CPU workers with batch size 64 each. This notebook follows that design. It asks whether four CPUs improve throughput, but the distributed global batch becomes 256.
- **Optimization-equivalent comparison:** keep the global batch at 64 by using 16 samples per worker. This preserves the number of optimizer updates but introduces four times as many distributed synchronization points as the first design.

When increasing global batch size, practitioners often scale the learning rate and then validate convergence. For example, linear scaling would change 0.01 to 0.04 for four workers, but this is a modeling choice rather than a timing requirement. Always compare both throughput (samples per second) and validation quality—not wall-clock time alone.

### When Ray Train is more likely to help

Distributed data-parallel training becomes more compelling when each synchronization is backed by substantial computation: larger models, larger batches, expensive preprocessing, longer runs, multiple GPUs, or multiple machines. On one CPU machine, a well-tuned multithreaded PyTorch process can be more efficient than several synchronized processes. Ray still adds value for cluster orchestration, fault handling, experiment scaling, and integration with distributed data pipelines, even when this tiny benchmark does not run faster.

## Conclusion

This experiment demonstrates correct synchronous data-parallel training: Ray partitions FashionMNIST across workers, DDP maintains identical model replicas, and gradients are combined after every backward pass. It also shows why distributed execution is not automatically faster. For a small CPU model, process startup and gradient communication can consume the time saved by processing data shards concurrently.

The revised benchmark removes repeated downloads, constrains CPU threads, reports global batch size, measures both paths explicitly, and explains the remaining semantic difference in optimizer steps. To observe clearer scaling, repeat the experiment with a larger convolutional network, more epochs, or one GPU per worker, and report samples per second together with validation accuracy.